# Ejercicio Integrador

Primero analizaremos el ramo de accidentes personales

In [37]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Cargamos los datos


df = pd.read_parquet("C:\\Users\\gusta\\OneDrive\\Documentos\\Diplomado\\github\\diplomado-ml-gustavo-rodriguez-ayala\\Modulo_2\\cartera_q1_2026_final.parquet")
df = df[df["ramo"] == "Accidentes Personales"]

In [38]:
# Estadisticas descriptivas: Número de pólizas, prima total, número de siniestros, prima promedio, siniestro promedio, etc.
num_polizas = df['id_poliza'].nunique()
prima_total = df['prima_total'].sum()
num_siniestros = df['n_siniestros'].sum()
prima_promedio = df['prima_total'].mean()
siniestro_promedio = df['monto_pagado'].mean()

print(f"Número de pólizas: {num_polizas:,.0f}")
print(f"Prima total: {prima_total:,.2f}")
print(f"Número de siniestros: {num_siniestros:,.0f}")
print(f"Prima promedio: {prima_promedio:,.2f}")
print(f"Siniestro promedio: {siniestro_promedio:,.2f}")

Número de pólizas: 5,207
Prima total: 24,980,678.88
Número de siniestros: 1,833
Prima promedio: 4,797.52
Siniestro promedio: 4,284.89


In [40]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# Dashboard de distribuciones

fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Distribución por Sexo',
        'Distribución por Edad',
        'Distribución por Año de Inicio',
        'Distribución por Plan',
        'Distribución de Prima Total',
        'Distribución de Monto Pagado'
    ),
    vertical_spacing=0.15,
    horizontal_spacing=0.08
)

ancho_barras = 0.5

sexo_counts = df['sexo'].value_counts()
fig.add_trace(
    go.Bar(x=sexo_counts.index, y=sexo_counts.values, marker_color='#3498db', name='Sexo', width=ancho_barras),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=df['edad'], nbinsx=30, marker_color='#2ecc71', name='Edad', opacity=0.8),
    row=1, col=2
)

anio_counts = df['anio_poliza'].value_counts().sort_index()
fig.add_trace(
    go.Bar(x=anio_counts.index, y=anio_counts.values, marker_color='#9b59b6', name='Año', width=ancho_barras),
    row=1, col=3
)

plan_counts = df['plan'].value_counts()
fig.add_trace(
    go.Bar(x=plan_counts.index, y=plan_counts.values, marker_color='#f1c40f', name='Plan', width=ancho_barras),
    row=2, col=1
)

fig.add_trace(
    go.Histogram(x=df['prima_total'], nbinsx=30, marker_color='#e67e22', name='Prima Total', opacity=0.8),
    row=2, col=2
)

fig.add_trace(
    go.Histogram(x=df['monto_pagado'], nbinsx=30, marker_color='#e74c3c', name='Monto Pagado', opacity=0.8),
    row=2, col=3
)

fig.update_layout(
    title_text="Dashboard Interactivo de Pólizas y Siniestros",
    title_x=0.5, 
    title_font=dict(size=24, color='#2c3e50', family="Arial, sans-serif"),
    showlegend=False, 
    height=800,
    width=1200,
    plot_bgcolor='rgba(245, 246, 250, 1)', 
    paper_bgcolor='white',
    font=dict(family="Arial, sans-serif", size=12, color='#34495e'),
    hovermode="closest",
    bargap=0.5
)

fig.update_xaxes(title_text="Sexo", row=1, col=1)
fig.update_yaxes(title_text="Cantidad", row=1, col=1)

fig.update_xaxes(title_text="Edad", row=1, col=2)
fig.update_yaxes(title_text="Frecuencia", row=1, col=2)

fig.update_xaxes(title_text="Año", type='category', row=1, col=3) 
fig.update_yaxes(title_text="Cantidad", row=1, col=3)

fig.update_xaxes(title_text="Plan", row=2, col=1)
fig.update_yaxes(title_text="Cantidad", row=2, col=1)

fig.update_xaxes(title_text="Monto ($)", row=2, col=2)
fig.update_yaxes(title_text="Frecuencia", row=2, col=2)

fig.update_xaxes(title_text="Monto ($)", row=2, col=3)
fig.update_yaxes(title_text="Frecuencia", row=2, col=3)

# Mostrar figura
fig.show()

El portafolio de Accidentes Personales se concentra en productos empaquetados (Individual y Familiar), pues son los planes con los que cuenta; con una tarifación estructurada en tres niveles de costo bien segmentados, como se puede observar en la gráfica. El comportamiento de los siniestros es el esperado para el ramo: muy baja frecuencia y severidad topada. Notamos que tenemos una distribución casi uniforme para la edad.

In [48]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# Supongamos que tu dataframe es 'df'
df_encoded = df.copy()

cols_drop = [
    'id_poliza', 'num_poliza', 'nombre', 'apellido_paterno', 'apellido_materno', 
    'rfc', 'agente_id', 'nombre_agente', 'fecha_emision', 'fecha_inicio_vigencia', 
    'fecha_fin_vigencia', 'cod_ramo', 'ramo_desde_codigo','marca_vehiculo', 'modelo_vehiculo'
]
df_encoded.drop(columns=cols_drop, inplace=True, errors='ignore')

# LABEL ENCODING (Para ordinales y categóricas de alta cardinalidad)
label_cols = [
    'nivel_riesgo', 'cuartil_prima', 'segmento_prima_fijo', # Ordinales
    'ocupacion', 'estado', 'municipio', 'codigo_postal',    # Alta cardinalidad
    'plan', 
    'motivo_baja', 'tipo_principal'
]

le = LabelEncoder()
for col in label_cols:
    if col in df_encoded.columns:
        # Convertir a string para evitar errores con nulos mezclados
        df_encoded[col] = df_encoded[col].astype(str)
        df_encoded[col] = le.fit_transform(df_encoded[col])

# 3. ONE-HOT ENCODING (Para nominales de baja cardinalidad)
ohe_cols = [
    'sexo', 'estado_civil', 'status_poliza', 'canal_venta', 'forma_pago', 
    'tipo_agente', 'tipo_vehiculo', 'tiene_abierto'
]

# sparse_output=False devuelve array numpy; drop='first' evita redundancia lineal
ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
ohe_arrays = ohe.fit_transform(df_encoded[ohe_cols])

# Reconstruir el DataFrame de las variables One-Hot
ohe_df = pd.DataFrame(
    ohe_arrays, 
    columns=ohe.get_feature_names_out(ohe_cols), 
    index=df_encoded.index
)

# Integrar todo
df_encoded = pd.concat([df_encoded.drop(columns=ohe_cols), ohe_df], axis=1)

df_encoded

,edad,ocupacion,ramo,plan,num_renovaciones,motivo_baja,suma_asegurada,deducible,prima_neta,prima_total,...,canal_venta_Broker,canal_venta_Digital,canal_venta_Directo,canal_venta_Promotor,forma_pago_Mensual,forma_pago_Semestral,forma_pago_Trimestral,tipo_agente_Independiente,tipo_agente_Promotor,tiene_abierto_True
15,65,12,Accidentes Personales,0,1,4,300000,3000.0,3600.0,4301.28,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
20,28,19,Accidentes Personales,1,0,4,500000,3000.0,6000.0,7516.80,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
33,40,19,Accidentes Personales,0,2,4,300000,2000.0,3600.0,4384.80,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
34,41,3,Accidentes Personales,1,0,4,300000,1000.0,3600.0,4176.00,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
36,22,5,Accidentes Personales,0,1,4,300000,2000.0,3600.0,4176.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49989,45,3,Accidentes Personales,0,1,4,300000,1000.0,3600.0,4301.28,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
49993,21,11,Accidentes Personales,0,0,4,200000,1000.0,2400.0,2784.00,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
49994,52,9,Accidentes Personales,1,0,4,200000,2000.0,2400.0,2784.00,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
49995,25,2,Accidentes Personales,1,1,4,300000,3000.0,3600.0,4510.08,...,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
